In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict
from catboost import CatBoost, Pool

DATA_DIR = Path('/workspace/data/raw')
OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
N_CANDIDATES = 64

# ============================================================
# データ読み込み
# ============================================================
transactions_raw = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers_raw = pl.read_csv(
    DATA_DIR / 'customers.csv',
    schema_overrides={'customer_id': pl.String}
)

articles_raw = pl.read_csv(
    DATA_DIR / 'articles.csv',
    schema_overrides={'article_id': pl.String}
)

# ============================================================
# 整数変換マップ作成
# ============================================================
print("整数変換マップ作成中...")

# customer_id → 整数
all_customer_ids = pl.concat([
    transactions_raw.select('customer_id'),
    customers_raw.select('customer_id')
]).unique().sort('customer_id')['customer_id'].to_list()

customer_id_map = {v: i for i, v in enumerate(all_customer_ids)}
customer_id_reverse = {i: v for v, i in customer_id_map.items()}

# article_id → 整数
all_article_ids = pl.concat([
    transactions_raw.select('article_id'),
    articles_raw.select('article_id')
]).unique().sort('article_id')['article_id'].to_list()

article_id_map = {v: i for i, v in enumerate(all_article_ids)}
article_id_reverse = {i: v for v, i in article_id_map.items()}

print(f"customer数: {len(customer_id_map):,}")
print(f"article数: {len(article_id_map):,}")

# ============================================================
# 整数変換済みデータ作成
# ============================================================
print("整数変換中...")

transactions_full = (
    transactions_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32).alias('customer_id'),
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32).alias('article_id'),
    ])
    .drop_nulls()
)

customers_full = (
    customers_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32).alias('customer_id'),
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin'),
        pl.col('age').fill_null(-1).cast(pl.Int32),
        pl.col('FN').fill_null(0),
        pl.col('Active').fill_null(0),
    ])
    .drop_nulls(subset=['customer_id'])
)

articles_full = (
    articles_raw
    .with_columns(
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32).alias('article_id'),
    )
    .drop_nulls(subset=['article_id'])
)

# product_codeマッピング（整数ベース）
product_code_map = {}
article_to_product = {}
for row in articles_full.select(['article_id', 'product_code']).iter_rows(named=True):
    code = row['product_code']
    art = row['article_id']
    article_to_product[art] = code
    if code not in product_code_map:
        product_code_map[code] = []
    product_code_map[code].append(art)

print(f"transactions: {len(transactions_full):,}行")
print(f"customers: {len(customers_full):,}行")
print(f"articles: {len(articles_full):,}行")
print("整数変換完了!")

整数変換マップ作成中...
customer数: 1,371,980
article数: 105,542
整数変換中...
transactions: 31,788,324行
customers: 1,371,980行
articles: 105,542行
整数変換完了!


In [2]:
# ============================================================
# 共通関数
# ============================================================
feature_cols = [
    'rank', 'score', 'is_history', 'is_variant',
    'age', 'days_since_last_purchase', 'repeat_count',
    'product_type_no', 'garment_group_no',
    'week0_sales', 'week1_sales',
    'age_bin_purchase_rate', 'days_since_last_product_type',
]

def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def create_features_light(transactions_full, customers_full, articles_full, cutoff, candidates_df):
    print("  特徴量作成中...")
    df_train = transactions_full.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # repeat_count
    repeat_count = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('repeat_count'))
    )
    
    # days_since_last_purchase
    user_last = (
        df_train
        .group_by('customer_id')
        .agg(pl.col('t_dat').max().alias('last_date'))
    ).with_columns(
        (pl.lit(cutoff) - pl.col('last_date')).dt.total_days().alias('days_since_last_purchase')
    ).select(['customer_id', 'days_since_last_purchase'])
    
    # article属性
    article_attrs = articles_full.select([
        'article_id', 'product_type_no', 'garment_group_no'
    ])
    
    # 週別販売数（直近2週）
    week0_sales = (
        df_train
        .filter(pl.col('t_dat') > pl.lit(cutoff - timedelta(days=7)))
        .group_by('article_id')
        .agg(pl.len().alias('week0_sales'))
    )
    week1_sales = (
        df_train
        .filter(
            (pl.col('t_dat') > pl.lit(cutoff - timedelta(days=14))) &
            (pl.col('t_dat') <= pl.lit(cutoff - timedelta(days=7)))
        )
        .group_by('article_id')
        .agg(pl.len().alias('week1_sales'))
    )
    
    # 年齢ビン別購買率
    age_bin_counts = (
        df_train
        .join(customers_full.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by(['age_bin', 'article_id'])
        .agg(pl.len().alias('age_bin_article_count'))
    )
    age_bin_totals = (
        df_train
        .join(customers_full.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by('age_bin')
        .agg(pl.len().alias('age_bin_total'))
    )
    age_bin_purchase_rate = (
        age_bin_counts
        .join(age_bin_totals, on='age_bin', how='left')
        .with_columns(
            (pl.col('age_bin_article_count') / pl.col('age_bin_total')).alias('age_bin_purchase_rate')
        )
        .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
    )
    
    # product_type別最終購買日
    tx_with_attrs = df_train.join(
        articles_full.select(['article_id', 'product_type_no']),
        on='article_id', how='left'
    )
    last_purchase_by_product_type = (
        tx_with_attrs
        .group_by(['customer_id', 'product_type_no'])
        .agg(pl.col('t_dat').max().alias('last_purchase_product_type'))
    )
    
    # 結合
    result = candidates_df.clone()
    result = result.join(article_attrs, on='article_id', how='left')
    result = result.join(
        customers_full.select(['customer_id', 'age', 'age_bin']),
        on='customer_id', how='left'
    )
    result = result.join(user_last, on='customer_id', how='left')
    result = result.join(repeat_count, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    result = result.join(week0_sales, on='article_id', how='left').with_columns(
        pl.col('week0_sales').fill_null(0)
    )
    result = result.join(week1_sales, on='article_id', how='left').with_columns(
        pl.col('week1_sales').fill_null(0)
    )
    result = result.join(
        age_bin_purchase_rate,
        on=['age_bin', 'article_id'], how='left'
    ).with_columns(pl.col('age_bin_purchase_rate').fill_null(0))
    result = result.join(
        last_purchase_by_product_type,
        on=['customer_id', 'product_type_no'], how='left'
    ).with_columns(
        pl.when(pl.col('last_purchase_product_type').is_not_null())
        .then((pl.lit(cutoff) - pl.col('last_purchase_product_type')).dt.total_days())
        .otherwise(-1)
        .alias('days_since_last_product_type')
    ).drop('last_purchase_product_type')
    
    result = result.with_columns([
        pl.col('age').fill_null(-1),
        pl.col('days_since_last_purchase').fill_null(-1),
        pl.col('product_type_no').fill_null(-1),
        pl.col('garment_group_no').fill_null(-1),
        pl.col('days_since_last_product_type').fill_null(-1),
    ])
    
    print(f"  特徴量作成完了: {result.shape}")
    return result

def create_reranking_dataset(transactions_full, customers_full, articles_full, cutoff,
                              label_days=7, n_candidates=32):
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions_full, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    
    df_train = transactions_full.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 時間減衰スコア
    df_train_td = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train_td.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train_td = df_train_td.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train_td = (
        df_train_td
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    purchase_scores = (
        df_train_td.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    general_pred = (
        df_train_td.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(n_candidates)
        .select('article_id').to_series().to_list()
    )
    
    # timedecay候補
    timedecay_candidates = (
        purchase_scores
        .filter(pl.col('customer_id').is_in(valid_customers_list))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= n_candidates)
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.UInt32),
            pl.lit(1).alias('is_history'),
            pl.lit(0).alias('is_variant'),
        ])
    )
    
    # variant候補
    articles_product = articles_full.select(['article_id', 'product_code'])
    variants_df = (
        timedecay_candidates.select(['customer_id', 'article_id'])
        .join(articles_product, on='article_id', how='left')
        .join(
            articles_product.rename({'article_id': 'variant_article_id'}),
            on='product_code', how='left'
        )
        .filter(pl.col('article_id') != pl.col('variant_article_id'))
        .select(['customer_id', pl.col('variant_article_id').alias('article_id')])
        .unique()
        .join(
            timedecay_candidates.select(['customer_id', 'article_id']),
            on=['customer_id', 'article_id'],
            how='anti'
        )
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.lit(0.0).alias('score'),
            pl.lit(999).cast(pl.UInt32).alias('rank'),
            pl.lit(0).alias('is_history'),
            pl.lit(1).alias('is_variant'),
        ])
    )
    
    # bin_popular候補
    customers_age = customers_full.select(['customer_id', 'age_bin'])
    recent = (
        df_train
        .filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
        .join(customers_age, on='customer_id', how='left')
    )
    bin_popular = (
        recent
        .group_by(['age_bin', 'article_id'])
        .agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('age_bin').alias('bin_rank')
        )
        .filter(pl.col('bin_rank') <= n_candidates)
        .select(['age_bin', 'article_id'])
    )
    
    valid_customers_df = pl.DataFrame({'customer_id': valid_customers_list}).with_columns(
        pl.col('customer_id').cast(pl.Int32)
    )
    
    existing_td_var = pl.concat([
        timedecay_candidates.select(['customer_id', 'article_id']),
        variants_df.select(['customer_id', 'article_id'])
    ]).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
    ])
    
    bin_pop_expanded = (
        valid_customers_df
        .join(customers_age, on='customer_id', how='left')
        .join(bin_popular, on='age_bin', how='left')
        .select(['customer_id', 'article_id'])
        .drop_nulls()
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
        ])
        .join(existing_td_var, on=['customer_id', 'article_id'], how='anti')
        .with_columns([
            pl.lit(0.0).alias('score'),
            pl.lit(998).cast(pl.UInt32).alias('rank'),
            pl.lit(0).alias('is_history'),
            pl.lit(0).alias('is_variant'),
        ])
    )
    
    # general_pred候補
    general_pred_df = pl.DataFrame({
        'article_id': general_pred,
    }).with_columns(pl.col('article_id').cast(pl.Int32))
    
    existing_all = pl.concat([
        existing_td_var,
        bin_pop_expanded.select(['customer_id', 'article_id'])
    ]).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
    ])
    
    gen_expanded = (
        valid_customers_df
        .join(general_pred_df, how='cross')
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
        ])
        .join(existing_all, on=['customer_id', 'article_id'], how='anti')
        .with_columns([
            pl.lit(0.0).alias('score'),
            pl.lit(997).cast(pl.UInt32).alias('rank'),
            pl.lit(0).alias('is_history'),
            pl.lit(0).alias('is_variant'),
        ])
    )
    
    # 全候補を結合してN件に絞る
    all_candidates = (
        pl.concat([
            timedecay_candidates.select(['customer_id', 'article_id', 'score', 'rank', 'is_history', 'is_variant']),
            variants_df.select(['customer_id', 'article_id', 'score', 'rank', 'is_history', 'is_variant']),
            bin_pop_expanded.select(['customer_id', 'article_id', 'score', 'rank', 'is_history', 'is_variant']),
            gen_expanded.select(['customer_id', 'article_id', 'score', 'rank', 'is_history', 'is_variant']),
        ])
        .unique(subset=['customer_id', 'article_id'])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('final_rank')
        )
        .filter(pl.col('final_rank') <= n_candidates)
        .with_columns(pl.col('final_rank').cast(pl.UInt32).alias('rank'))
        .drop('final_rank')
    )
    
    # labelを付与
    actuals_df = pl.DataFrame([
        {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
        for cid, aids in actuals.items()
        for aid in aids
    ]).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
    ])
    
    all_candidates = (
        all_candidates
        .join(actuals_df, on=['customer_id', 'article_id'], how='left')
        .with_columns(pl.col('label').fill_null(0))
    )
    
    print(f"  候補数: {len(all_candidates):,}")
    print(f"  正例数: {all_candidates['label'].sum():,}")
    
    result = create_features_light(
        transactions_full, customers_full, articles_full,
        cutoff, all_candidates
    )
    
    return result

print("関数定義完了")

関数定義完了


In [3]:
train_cutoffs = [
    date(2020, 9, 8),   # valid: 9/9〜9/15
    date(2020, 9, 1),   # valid: 9/2〜9/8
    date(2020, 8, 25),  # valid: 8/26〜9/1
    date(2020, 8, 18),  # valid: 8/19〜8/25
    date(2020, 8, 11),  # valid: 8/12〜8/18
]

valid_cutoff = date(2020, 9, 15)  # valid: 9/16〜9/22

print("学習データ作成中（5fold）...")
train_dfs = []
for cutoff in train_cutoffs:
    print(f"\n{'='*40}")
    print(f"cutoff: {cutoff}")
    print(f"{'='*40}")
    df = create_reranking_dataset(
        transactions_full, customers_full, articles_full,
        cutoff, n_candidates=N_CANDIDATES
    )
    train_dfs.append(df.to_pandas())

train_df = pd.concat(train_dfs, ignore_index=True).sort_values('customer_id')
print(f"\n学習データ合計: {len(train_df):,}行")
print(f"正例数: {train_df['label'].sum():,}")
print(f"正例率: {train_df['label'].mean():.4f}")

学習データ作成中（5fold）...

cutoff: 2020-09-08

cutoff: 2020-09-08
  候補数: 4,609,216
  正例数: 15,135
  特徴量作成中...
  特徴量作成完了: (4609216, 17)

cutoff: 2020-09-01

cutoff: 2020-09-01
  候補数: 4,852,608
  正例数: 15,698
  特徴量作成中...
  特徴量作成完了: (4852608, 17)

cutoff: 2020-08-25

cutoff: 2020-08-25
  候補数: 5,136,192
  正例数: 15,139
  特徴量作成中...
  特徴量作成完了: (5136192, 17)

cutoff: 2020-08-18

cutoff: 2020-08-18
  候補数: 4,610,240
  正例数: 14,295
  特徴量作成中...
  特徴量作成完了: (4610240, 17)

cutoff: 2020-08-11

cutoff: 2020-08-11
  候補数: 4,550,016
  正例数: 15,081
  特徴量作成中...
  特徴量作成完了: (4550016, 17)

学習データ合計: 23,758,272行
正例数: 75,348
正例率: 0.0032


In [4]:
# CatBoost学習
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df['label'],
    group_id=train_df['customer_id']
)

model = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 200,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 20,
    'early_stopping_rounds': 20,
})

model.fit(train_pool)
print("学習完了")

0:	total: 4.81s	remaining: 15m 58s
20:	total: 57.5s	remaining: 8m 9s
40:	total: 1m 49s	remaining: 7m 5s
60:	total: 2m 41s	remaining: 6m 8s
80:	total: 3m 34s	remaining: 5m 15s
100:	total: 4m 26s	remaining: 4m 21s
120:	total: 5m 18s	remaining: 3m 28s
140:	total: 6m 11s	remaining: 2m 35s
160:	total: 7m 4s	remaining: 1m 42s
180:	total: 7m 58s	remaining: 50.2s
199:	total: 8m 48s	remaining: 0us
学習完了


In [ ]:
# ============================================================
# valid評価
# ============================================================
valid_cutoff = date(2020, 9, 8)
valid_df = create_reranking_dataset(
    transactions_full, customers_full, articles_full,
    valid_cutoff, n_candidates=N_CANDIDATES
).to_pandas()

valid_df['pred_score'] = model.predict(valid_df[feature_cols])

actuals_valid = get_actuals(transactions_full, valid_cutoff)
metric_acc = {k: {'customer_recall': [], 'map': []} for k in [12, 24, 32]}

for customer_id, group in valid_df.groupby('customer_id'):
    actual_set = actuals_valid.get(customer_id, set())
    if not actual_set:
        continue
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    for k in [12, 24, 32]:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['customer_recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("【valid評価】")
print(f"{'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in [12, 24, 32]:
    recall = np.mean(metric_acc[k]['customer_recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>4} {recall:>10.6f} {map_k:>10.6f}")


cutoff: 2020-09-08
  候補数: 4,609,216
  正例数: 15,224
  特徴量作成中...
  特徴量作成完了: (4609216, 17)
【valid評価】
   k     recall        MAP
------------------------------
  12   0.057282   0.030777
  24   0.069033   0.031492
  32   0.074023   0.031691


: 

In [ ]:
# ============================================================
# 提出ファイル作成（バッチ処理版）
# ============================================================
from tqdm import tqdm

cutoff = transactions_full['t_dat'].max()
print(f"提出用cutoff: {cutoff}")

df_train_all = transactions_full.filter(pl.col('t_dat') <= pl.lit(cutoff))

# 時間減衰スコア
print("時間減衰スコア計算中...")
df_train_td = df_train_all.with_columns(
    (pl.col('t_dat') +
     ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
     ).cast(pl.Int32) * pl.duration(days=1)
    ).alias('ldbw')
)
weekly_sales = df_train_td.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
df_train_td = df_train_td.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
last_week_sales = (
    weekly_sales.filter(pl.col('ldbw') == cutoff)
    .select(['article_id', 'count']).rename({'count': 'count_targ'})
)
df_train_td = (
    df_train_td
    .join(last_week_sales, on='article_id', how='left')
    .with_columns(pl.col('count_targ').fill_null(0))
    .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
    .with_columns(
        (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
    )
    .with_columns(
        (
            pl.col('quotient') * (
                a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
            ).clip(lower_bound=0)
        ).alias('score')
    )
)

# timedecay候補
print("timedecay候補生成中...")
timedecay_candidates_all = (
    df_train_td.group_by(['customer_id', 'article_id'])
    .agg(pl.col('score').sum())
    .filter(pl.col('score') > 0)
    .with_columns(
        pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
    )
    .filter(pl.col('rank') <= N_CANDIDATES)
    .with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.UInt32),
        pl.lit(1).alias('is_history'),
        pl.lit(0).alias('is_variant'),
    ])
)
print(f"  timedecay候補数: {len(timedecay_candidates_all):,}")

# general_pred
general_pred = (
    df_train_td.group_by('article_id')
    .agg(pl.col('quotient').sum().alias('total_quotient'))
    .sort('total_quotient', descending=True)
    .head(N_CANDIDATES)
    .select('article_id').to_series().to_list()
)

# variant候補
print("variant候補生成中...")
articles_product = articles_full.select(['article_id', 'product_code'])
variants_all = (
    timedecay_candidates_all.select(['customer_id', 'article_id'])
    .join(articles_product, on='article_id', how='left')
    .join(
        articles_product.rename({'article_id': 'variant_article_id'}),
        on='product_code', how='left'
    )
    .filter(pl.col('article_id') != pl.col('variant_article_id'))
    .select(['customer_id', pl.col('variant_article_id').alias('article_id')])
    .unique()
    .join(
        timedecay_candidates_all.select(['customer_id', 'article_id']),
        on=['customer_id', 'article_id'], how='anti'
    )
    .with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.lit(0.0).alias('score'),
        pl.lit(999).cast(pl.UInt32).alias('rank'),
        pl.lit(0).alias('is_history'),
        pl.lit(1).alias('is_variant'),
    ])
)
print(f"  variant候補数: {len(variants_all):,}")

# bin_popular候補
print("bin_popular候補生成中...")
customers_age = customers_full.select(['customer_id', 'age_bin'])
recent = (
    df_train_all
    .filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    .join(customers_age, on='customer_id', how='left')
)
bin_popular = (
    recent
    .group_by(['age_bin', 'article_id'])
    .agg(pl.len().alias('count'))
    .with_columns(
        pl.col('count').rank(method='ordinal', descending=True).over('age_bin').alias('bin_rank')
    )
    .filter(pl.col('bin_rank') <= N_CANDIDATES)
    .select(['age_bin', 'article_id'])
)

# age_binマップ
age_bin_dict = dict(zip(
    customers_full['customer_id'].to_list(),
    customers_full['age_bin'].to_list()
))

# general_predのset
general_pred_set = set(general_pred)

# バッチ処理で全ユーザーの候補を生成
print("全ユーザーの候補生成中（バッチ処理）...")
BATCH_SIZE = 100000

all_customer_ids = list(customer_id_map.values())
all_rows = []

# timedecayとvariantをdictに変換
td_dict = {}
for row in timedecay_candidates_all.iter_rows(named=True):
    cid = row['customer_id']
    if cid not in td_dict:
        td_dict[cid] = []
    td_dict[cid].append((row['article_id'], row['score'], row['rank']))

var_dict = {}
for row in variants_all.iter_rows(named=True):
    cid = row['customer_id']
    if cid not in var_dict:
        var_dict[cid] = []
    var_dict[cid].append(row['article_id'])

# bin_popularをdictに変換
bin_pop_dict = {}
for row in bin_popular.iter_rows(named=True):
    ab = row['age_bin']
    if ab not in bin_pop_dict:
        bin_pop_dict[ab] = []
    bin_pop_dict[ab].append(row['article_id'])

for batch_start in tqdm(range(0, len(all_customer_ids), BATCH_SIZE), desc="バッチ処理"):
    batch_customers = all_customer_ids[batch_start:batch_start+BATCH_SIZE]
    batch_rows = []
    
    for cid in batch_customers:
        td_cands = td_dict.get(cid, [])
        var_cands = var_dict.get(cid, [])
        age_bin = age_bin_dict.get(cid, -1)
        bin_pop_cands = bin_pop_dict.get(age_bin, [])
        
        existing = set([a for a, _, _ in td_cands])
        
        candidates = []
        # timedecay
        for art, score, rank in td_cands:
            candidates.append((art, score, rank, 1, 0))
        
        # variant
        for art in var_cands:
            if art not in existing:
                candidates.append((art, 0.0, 999, 0, 1))
                existing.add(art)
        
        # bin_popular
        for art in bin_pop_cands:
            if art not in existing:
                candidates.append((art, 0.0, 998, 0, 0))
                existing.add(art)
        
        # general_pred
        for art in general_pred:
            if art not in existing:
                candidates.append((art, 0.0, 997, 0, 0))
                existing.add(art)
        
        candidates = candidates[:N_CANDIDATES]
        
        for art, score, rank, is_hist, is_var in candidates:
            batch_rows.append({
                'customer_id': cid,
                'article_id': art,
                'score': score,
                'rank': rank,
                'is_history': is_hist,
                'is_variant': is_var,
            })
    
    all_rows.extend(batch_rows)

print(f"全候補数: {len(all_rows):,}")

# DataFrameに変換
print("DataFrame変換中...")
all_candidates_df = pl.DataFrame(all_rows).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
    pl.col('rank').cast(pl.UInt32),
])

# 特徴量作成
print("特徴量作成中...")
all_features_df = create_features_light(
    transactions_full, customers_full, articles_full,
    cutoff, all_candidates_df
).to_pandas()

# 予測
print("予測中...")
all_features_df['pred_score'] = model.predict(all_features_df[feature_cols])

# 上位12件を選択
print("上位12件選択中...")
submission = (
    all_features_df
    .sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    .groupby('customer_id')
    .head(12)
    .groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join([article_id_reverse[i] for i in x]))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# customer_idを元の文字列に戻す
submission['customer_id'] = submission['customer_id'].map(customer_id_reverse)

# 全ユーザーにマージ
general_pred_top12 = ' '.join([article_id_reverse[i] for i in general_pred[:12]])
all_customers_final = pd.DataFrame({'customer_id': list(customer_id_map.keys())})
all_customers_final = all_customers_final.merge(submission, on='customer_id', how='left')
all_customers_final['prediction'] = all_customers_final['prediction'].fillna(general_pred_top12)

# 保存
all_customers_final.to_csv(OUTPUT_DIR / 'submission_rerank_v4.csv', index=False)
print(f"保存完了: {OUTPUT_DIR / 'submission_rerank_v4.csv'}")
print(f"ユーザー数: {len(all_customers_final):,}")
print(all_customers_final.head(5))

提出用cutoff: 2020-09-22
時間減衰スコア計算中...
timedecay候補生成中...
  timedecay候補数: 9,891,933
variant候補生成中...
  variant候補数: 66,810,920
bin_popular候補生成中...
全ユーザーの候補生成中（バッチ処理）...


バッチ処理:  86%|████████▌ | 12/14 [02:02<00:23, 11.79s/it]